# 08_adipose_curation

Selected source cells for review and reconstruction.

Source: `Adipose_combined_samples_CellSegmentation/Adipose_Combination_5(label transfer+Cluster naming).ipynb`

Uses original Cellname_final labels, not later experimental Cellname_final_RNA/SCT. Only the resolution 0.20 column is used for final annotation.

Original execution counts and plots were removed. Do not run in the source-data folder: original output filenames and Docker paths are retained as provenance. Use the common scripts for new runs. These excerpts have not been executed as a fresh notebook.


Source cell 1 (one-based); original execution count 1.

In [ ]:
library(Seurat)
library(spacexr)

Source cell 2 (one-based); original execution count 2.

In [ ]:
merge.obj<-readRDS(
  file = "adipose_mergeobj_preProject_sketchHarmony_labelTransferred.rds")

Source cell 17 (one-based); original execution count 20.

In [ ]:
DefaultAssay(merge.obj) <- "RNA"

merge.obj <- SCTransform(
  merge.obj,
  assay = "RNA",
  new.assay.name = "SCT",
  verbose = FALSE
)

Source cell 27 (one-based); original execution count 33.

In [ ]:
# Cellname_partial を作成（Cellname_final をベースに一部まとめる）
# 前提：merge.obj の meta.data に Cellname_final がある

merge.obj$Cellname_partial <- as.character(merge.obj$Cellname_final)

# 置換ルール
merge.obj$Cellname_partial[merge.obj$Cellname_partial == "Adjacent parenchymal cells"] <- "Unknown"
merge.obj$Cellname_partial[merge.obj$Cellname_partial == "Endothelial cell"]           <- "Endothelial_mixed"
merge.obj$Cellname_partial[merge.obj$Cellname_partial == "Macrophage/Monocyte/DC"]     <- "Myeloid_mixed"
# Unknown は Unknown のまま（何もしない）

# factor化（順序は必要ならここで指定）
merge.obj$Cellname_partial <- factor(merge.obj$Cellname_partial)

# 取りこぼし（想定外ラベル）がないか確認：0 ならOK
setdiff(unique(as.character(merge.obj$Cellname_final)),
        c("Adipocyte","Adipose stromal cell","Adjacent parenchymal cells","B cell",
          "Endothelial cell","Epithelial cell","Erythroid cell","Macrophage/Monocyte/DC",
          "Mesothelial cell","Neutrophil","NK cell","T cell","Unknown","VSMC"))


Source cell 36 (one-based); original execution count 42.

In [ ]:
merge.obj <- FindNeighbors(
  merge.obj,
  reduction  = "harmony",
  dims       = 1:30,
  k.param    = 50,
  graph.name = "harmony_snn_k50"
)

merge.obj <- FindClusters(
  merge.obj,
  graph.name  = "harmony_snn_k50",
  resolution  = 0.20,
  random.seed = 1
)

Source cell 42 (one-based); original execution count 53.

In [ ]:
# 0.20クラスタ列を明示的に使う（表示用）
merge.obj$clusters_k50_r020 <- merge.obj[[]][["harmony_snn_k50_res.0.2"]]

# UMAP表示（0.20で色分け）
DimPlot(
  merge.obj,
  reduction = "umap",
  group.by  = "clusters_k50_r020",
  label     = TRUE,
  repel     = TRUE,
  raster    = TRUE
)


Source cell 45 (one-based); original execution count 56.

In [ ]:
# 前提：k50_r0.20 のクラスタ列を使う
cluster_col <- "harmony_snn_k50_res.0.2"
stopifnot(cluster_col %in% colnames(merge.obj@meta.data))

# cluster id（文字列）を取り出し
cl <- as.character(merge.obj@meta.data[[cluster_col]])

# まず全部 Unassigned
new_name <- rep("Unassigned", length(cl))

# 個別指定
new_name[cl == "0"] <- "Adipocyte (mature; lipid-signal mixed)"
new_name[cl == "1"] <- "Endothelial (capillary~general)"
new_name[cl == "2"] <- "Adipose stromal / Fibroblast"
new_name[cl == "3"] <- "Myeloid (Macrophage/DC; MHCII+; incl M2-like)"
new_name[cl == "4"] <- "Vascular mural (VSMC/Pericyte) + Endo mixed (Endo-mural mixed; suspicious)"
new_name[cl == "5"] <- "B cell"
new_name[cl == "8"] <- "Erythroid"

# 10〜29 をまとめて指定（存在するものだけ）
new_name[cl %in% as.character(10:29)] <- "Adipocyte-like / lipid-signal dominated"

# 追加：meta.data に保存
merge.obj$cluster_based_naming <- new_name

# 確認（クロス表）
table(merge.obj[[cluster_col]][,1], merge.obj$cluster_based_naming, useNA = "ifany")

Source cell 49 (one-based); original execution count 61.

In [ ]:
## ===============================
## Merged_cellname を作る
## - Cellname_partial が通常群ならそのままコピー
## - 3群 (Endothelial_mixed / Myeloid_mixed / Unknown) のみ
##   cluster_based_naming で置き換え
## ===============================

# まずはそのまま転記
merge.obj$Merged_cellname <- as.character(merge.obj$Cellname_partial)

# 置き換え対象（3群）
groups_change <- c("Endothelial_mixed", "Myeloid_mixed", "Unknown")
idx_change <- which(as.character(merge.obj$Cellname_partial) %in% groups_change)

# cluster_based_naming -> Merged_cellname の対応表
map_cb_to_merged <- c(
  "Adipocyte (mature; lipid-signal mixed)" = "Adipocyte",
  "Adipocyte-like / lipid-signal dominated" = "Adipocyte",
  "Adipose stromal / Fibroblast" = "Adipose stromal cell",
  "B cell" = "B cell",
  "Endothelial (capillary~general)" = "Endothelial cell",
  "Erythroid" = "Erythroid cell",
  "Myeloid (Macrophage/DC; MHCII+; incl M2-like)" = "Macrophage / Monocyte / DC",
  "Unassigned" = "Unknown",
  "Vascular mural (VSMC/Pericyte) + Endo mixed (Endo-mural mixed; suspicious)" = "VSMC/Pericyte"
)

# 置き換え（該当しない/NA は Unknown に倒す）
cb <- as.character(merge.obj$cluster_based_naming[idx_change])
new_label <- unname(map_cb_to_merged[cb])
new_label[is.na(new_label) | new_label == ""] <- "Unknown"

merge.obj$Merged_cellname[idx_change] <- new_label
merge.obj$Merged_cellname <- factor(merge.obj$Merged_cellname)

# 軽い確認（3群だけ）
table(merge.obj$Cellname_partial[idx_change], merge.obj$Merged_cellname[idx_change])

Source cell 61 (one-based); original execution count 74.

In [ ]:
saveRDS(merge.obj, file = "adipose_mergeobj_sketchHarmony_SCT_labelTransferred_k50_r020_Naming_260220.rds")